In [5]:
import pandas as pd
from pathlib import Path
import pytz
import numpy as np
import datetime
import gc


In [6]:
campd_type_dict = {
    'State': 'string',
    'Facility Name': 'string',
    'Facility ID': 'Int32',
    'Unit ID': 'string',
    'Hour': 'Int8',
    'Operating Time': 'float32',
    'Gross Load (MW)': 'float32',
    'Steam Load (1000 lb/hr)': 'float32',
    'SO2 Mass (lbs)': 'float64',
    'SO2 Mass Measure Indicator': 'string',
    'SO2 Rate (lbs/mmBtu)': 'float32',
    'SO2 Rate Measure Indicator': 'string',
    'CO2 Mass (short tons)': 'float32',
    'CO2 Mass Measure Indicator': 'string',
    'CO2 Rate (short tons/mmBtu)': 'float32',
    'CO2 Rate Measure Indicator': 'string',
    'NOx Mass (lbs)': 'float32',
    'NOx Mass Measure Indicator': 'string',
    'NOx Rate (lbs/mmBtu)': 'float32',
    'NOx Rate Measure Indicator': 'string',
    'Heat Input (mmBtu)': 'float64',
    'Heat Input Measure Indicator': 'string',
    'Primary Fuel Type': 'string',
    'Secondary Fuel Type': 'string',
    'Unit Type': 'string',
    'SO2 Controls': 'string',
    'NOx Controls': 'string',
    'Program Code': 'string',
    'Year': 'int16',
    'Source Category': 'string',
    'Latitude': 'float32',
    'Longitude': 'float32',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float32',
    'Associated Generators & Nameplate Capacity (MWe)': 'string',
    'Nameplate Capacity (MWe)': 'float32',
    'Month': 'Int8',
    'hourly_capacity_factor': 'float32',
    'hourly_ramping_factor': 'float32',
    'Nameplate Capacity (MW) - EIA': 'float32',
    'Summer Capacity (MW)': 'float32',
    'Winter Capacity (MW)': 'float32',
    'Minimum Load (MW)': 'float32',
    'Operating Year_min': 'float32',
    'Operating Year_max': 'float32',
    'Operating Year_mean': 'float32',
    'Planned Retirement Year': 'Int16',
    'unit_age_min': 'Int8',
    'unit_age_max': 'Int8',
    'unit_age_mean': 'float32',
    'Day': 'float32'
}

<h1>Transform and Concat Temp Data</h1>

For each state we:
- read in the yearly data and concat it
- replace missing value indicators with nan
- filter out stations that started reporting after our start date (01/01/2019) to ensure consistency
- remove unrealistic values (very likely wrong measures)
- interpolate all nan temps linearly (using time interpolation)
- round timestamp to the lower hour (values are collected at hourly intervals, but not always precisely on the hour)
- adjust timezones of temp data to be consistent with those of CAMPD (transform local daylight savings time of ASOS to a fixed summertime like in CAMPD)
- compute the closest weather station for each facility using their coordinates (brute for with euclidean works at this scale)
- merge this data into the full campd dataset


<h2>NY</h2>

In [ ]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/NY')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/NY/Concatted File/temp_2019_2024_test.csv')
temp_df.to_csv(output_path, index=False)

In [ ]:
temp_df = pd.read_csv("Temp Data/NY/Concatted File/temp_2019_2024.csv")

In [ ]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})

In [ ]:
# remove any duplicate entries (we assume to be data issues)
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"], keep='first')
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,SWF,2019-01-01 00:45,-74.1048,41.5041,42.8
1,ALB,2019-01-01 00:51,-73.8036,42.7576,41.0
2,SLK,2019-01-01 00:51,-74.2062,44.3853,37.0
3,LGA,2019-01-01 00:51,-73.8803,40.7794,47.0
4,JFK,2019-01-01 00:51,-73.7622,40.6386,49.0
...,...,...,...,...,...
2352641,JHW,2023-12-31 23:56,-79.2580,42.1534,32.0
2352642,ELZ,2023-12-31 23:56,-77.9842,42.1078,33.0
2352643,ITH,2023-12-31 23:56,-76.4584,42.4910,32.0
2352644,ISP,2023-12-31 23:56,-73.1017,40.7939,39.0


In [231]:
# replace missing "M" with nan
temp_df.loc[temp_df["Temperature (F)"] == "M", "Temperature (F)"] = np.nan

In [232]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:.2f}")
temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

count   2233136.00
mean         50.51
std          18.70
min         -32.80
25%          36.00
50%          51.00
75%          66.00
max         141.80
Name: Temperature (F), dtype: float64

In [235]:
# remove unrealistic values for the state
temp_df.loc[temp_df["Temperature (F)"] > 105, "Temperature (F)"] = np.nan

In [236]:
temp_df.isna().sum()

Station                0
datetime               0
Longitude              0
Latitude               0
Temperature (F)    13971
dtype: int64

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [239]:
temp_df.isna().sum()

datetime           0
Station            0
Longitude          0
Latitude           0
Temperature (F)    0
dtype: int64

In [240]:
# round to the lower hour 
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [241]:
# transform EST/EDT timezone to naive EST (campd is in local standard time without daylight saving)

# localize to America/New_York 
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/New_York', ambiguous='NaT', nonexistent='shift_forward') # handle edge cases (ambigious time stamps fill with NaT -> when clock is moved backward, duplicate naive timestampts exist + non_existent time stamps are shifted forward to the next hour -> when clock is moved forward, there are missing naive time stamps)

# convert to fixed EST (UTC-5, no daylight saving)
fixed_est = pytz.timezone('Etc/GMT+5') # Note: "Etc/GMT+5" is actually UTC-5
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_est)

# remove timezone info to make it naive (EST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)

In [242]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [243]:
# check if any facilities changed location over the timeframe across all states

# group by Facility ID, then check if multiple unique coords exist
coord_changes = facility_df.groupby("Facility ID").agg({
    "Longitude": pd.Series.nunique,
    "Latitude": pd.Series.nunique
}).reset_index()

# Add a flag if yes
coord_changes['Has_Coordinate_Changed'] = (
    (coord_changes['Longitude'] > 1) | (coord_changes['Latitude'] > 1)
)

print(f"Number of facilities that changed coordinates: {coord_changes['Has_Coordinate_Changed'].sum()}")

# -> we can safely calculate distance for each facility without adjusting for time

Number of facilities that changed coordinates: 0


In [ ]:
facility_df = facility_df[facility_df["State"] == "NY"]
facility_df = facility_df[["State", "Facility Name", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

,State,Facility Name,Facility ID,Longitude,Latitude
950,NY,Danskammer Generating Station,2480,-73.9646,41.5730
954,NY,Arthur Kill,2490,-74.2027,40.5915
957,NY,East River,2493,-73.9742,40.7281
961,NY,Gowanus Generating Station,2494,-74.0051,40.6635
993,NY,Hudson Avenue,2496,-73.9807,40.7052
...,...,...,...,...,...
4110,NY,Valley Energy Center,56940,-74.4350,41.4130
4319,NY,Momentive Performance Materials,880024,-73.6728,42.8158
4343,NY,Lehigh Northeast Cement Company,880052,-73.6208,43.3064
4374,NY,Ravenswood Steam Plant,880100,-73.7500,40.7600


In [246]:
# calculate the nearest weather station for each facility using brute force euclidean distance

# Unique weather stations
stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

def find_nearest_station(row):
    dists = np.sqrt((stations['Longitude'] - row['Longitude'])**2 + 
                    (stations['Latitude'] - row['Latitude'])**2) 
    nearest_idx = dists.idxmin() # get index of nearest station (which aligns with the index in stations)
    # return a series with the nearest station info for the current row
    return pd.Series({
        "Facility ID": row['Facility ID'],
        "Facility Name": row['Facility Name'],
        "Longitude": row['Longitude'],
        "Latitude": row['Latitude'],
        'Nearest_Station': stations.loc[nearest_idx, 'Station'],
        'Station_Longitude': stations.loc[nearest_idx, 'Longitude'],
        'Station_Latitude': stations.loc[nearest_idx, 'Latitude']
    })

# apply to each unique facility location
unique_facilities = facility_df[["Facility Name", "Facility ID", "Longitude", "Latitude"]].drop_duplicates()
facility_df = unique_facilities.apply(find_nearest_station, axis=1)
facility_df["Facility ID"] = facility_df["Facility ID"].astype(int)  # ensure Facility ID is int

In [ ]:
campd_ny = pd.read_csv("NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_Full.csv", dtype=campd_type_dict)
campd_ny["datetime"] = pd.to_datetime(campd_ny["datetime"])

In [156]:
# Merge mapping into campd_ny
campd_ny = campd_ny.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

In [216]:
# Merge temp data using station name and timestamp
campd_ny_with_temp = campd_ny.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_ny_with_temp = campd_ny_with_temp.drop(columns=['Station'])

In [33]:
campd_ny_with_temp["Temperature (F)"].isna().sum()

283072

In [34]:
campd_ny_with_temp.to_csv("NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_Full.csv", index=False)

In [ ]:
del campd_ny, campd_ny_with_temp
gc.collect()

<h2>TX</h2>

In [ ]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/TX')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/TX/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [7]:
temp_df = pd.read_csv("Temp Data/TX/Concatted File/temp_2019_2024.csv")

In [ ]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})

# remove any duplicate entries (data issues)
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"], keep='first')
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,GZN,2024-01-01 00:00,-99.0237,32.3658,37.4
1,RYW,2024-01-01 00:15,-97.9659,30.4967,59.0
2,ADS,2024-01-01 00:35,-96.8364,32.9686,38.3
3,FTN,2024-01-01 00:40,-100.0188,28.2086,50.0
4,T35,2024-01-01 00:50,-96.9711,30.8794,50.4
...,...,...,...,...,...
10427431,OCH,2019-12-31 23:56,-94.7094,31.5778,36.0
10427432,BIF,2019-12-31 23:58,-106.3800,31.8495,37.6
10427433,GRK,2019-12-31 23:58,-97.8300,31.0700,46.8
10427434,HLR,2019-12-31 23:58,-97.7145,31.1387,44.6


In [7]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [8]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_14880\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   9426270.00
mean         68.29
std          17.16
min         -63.00
25%          57.00
50%          71.00
75%          80.60
max         149.00
Name: Temperature (F), dtype: float64

In [9]:
# lowest temperature ever recorded in Texas is -23 F, fill with NaN if lower than -20
temp_df.loc[temp_df["Temperature (F)"] < -20, "Temperature (F)"] = np.nan

# hottest official temperature recorded in Texas is 120, fill with NaN if higher than 120
temp_df.loc[temp_df["Temperature (F)"] > 120, "Temperature (F)"] = np.nan

In [ ]:
temp_df["Temperature (F)"].describe()

count   9421734.00
mean         68.27
std          17.10
min         -20.00
25%          57.00
50%          71.00
75%          80.60
max         119.00
Name: Temperature (F), dtype: float64

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [ ]:
# round to the lower hour 
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [13]:
# transform CST/CDT timezone to naive CST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Chicago timezone (handles CST/CDT)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Chicago', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed CST (UTC-6 without DST)
fixed_cst = pytz.FixedOffset(-6 * 60)  # UTC-6 for CST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_cst)

# Remove timezone information to make it naive (CST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [14]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [ ]:
# calculate the nearest weather station for each facility using brute force euclidean distance

# Unique weather stations
stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

def find_nearest_station(row):
    dists = np.sqrt((stations['Longitude'] - row['Longitude'])**2 + 
                    (stations['Latitude'] - row['Latitude'])**2) 
    nearest_idx = dists.idxmin() # get index of nearest station (which aligns with the index in stations)
    # return a series with the nearest station info for the current row
    return pd.Series({
        "Facility ID": row['Facility ID'],
        "Facility Name": row['Facility Name'],
        "Longitude": row['Longitude'],
        "Latitude": row['Latitude'],
        'Nearest_Station': stations.loc[nearest_idx, 'Station'],
        'Station_Longitude': stations.loc[nearest_idx, 'Longitude'],
        'Station_Latitude': stations.loc[nearest_idx, 'Latitude']
    })

# apply to each unique facility location
unique_facilities = facility_df[["Facility Name", "Facility ID", "Longitude", "Latitude"]].drop_duplicates()
facility_df = unique_facilities.apply(find_nearest_station, axis=1)
facility_df["Facility ID"] = facility_df["Facility ID"].astype(int)  # ensure Facility ID is int

In [16]:
facility_df

,State,Facility Name,Facility ID,Longitude,Latitude,Nearest_Station,Station_Longitude,Station_Latitude
0,TX,Copper Station,9,-106.38,31.76,ELP,-106.38,31.81
1,TX,Oklaunion Power Station,127,-99.18,34.08,F05,-99.28,34.23
2,TX,Limestone,298,-96.25,31.42,LXY,-96.51,31.64
3,TX,Laredo,3439,-99.51,27.57,LRD,-99.46,27.54
4,TX,Nueces Bay,3441,-97.42,27.82,CRP,-97.51,27.77
...,...,...,...,...,...,...,...,...
136,TX,SJRR Power Peaking Facility,62548,-95.11,29.88,IAH,-95.36,29.98
137,TX,Braes Bayou Plant,64383,-95.62,29.48,AXH,-95.48,29.51
138,TX,Mark One Power Station,65372,-95.43,29.23,LBX,-95.46,29.11
139,TX,Brotman Power Station,65373,-95.44,29.36,AXH,-95.48,29.51


<h3>NOx</h3>

we do it separately for each emission type due to file size (we use some of the downsampled data from the tx ML notebooks)

In [15]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/NOx/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

In [18]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])



In [25]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/NOx/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)

<h3>CO2</h3>

In [3]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/CO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

In [ ]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])

In [18]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/CO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)


<h3>SO2</h3>

In [ ]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/SO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

In [ ]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])

In [ ]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/SO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)


<h2>WY</h2>

In [ ]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/WY')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/WY/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [4]:
temp_df = pd.read_csv("Temp Data/WY/temp_2019_2024.csv")

In [ ]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})

# remove any duplicate entries (data issues)
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"], keep='first')
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,TBX,2024-01-01 00:50,-108.2384,43.4646,35.6
1,SHR,2024-01-01 00:53,-106.9700,44.7700,22.0
2,LAR,2024-01-01 00:53,-105.6750,41.3121,18.0
3,CPR,2024-01-01 00:53,-106.4644,42.9080,32.0
4,CYS,2024-01-01 00:53,-104.8105,41.1556,26.0
...,...,...,...,...,...
3179995,GEY,2021-11-06 21:53,-108.0828,44.5168,49.0
3179996,CPR,2021-11-06 21:53,-106.4644,42.9080,49.0
3179997,CYS,2021-11-06 21:53,-104.8105,41.1556,44.0
3179998,BPI,2021-11-06 21:53,-110.1111,42.5851,41.0


In [6]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [7]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_89308\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   2918074.00
mean         44.41
std          21.50
min         -42.00
25%          28.90
50%          43.00
75%          60.80
max         107.00
Name: Temperature (F), dtype: float64

In [ ]:
# values appear accurate, no unrealistic outliers

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [ ]:
# round to the lower hour 
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [10]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [11]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [ ]:
# calculate the nearest weather station for each facility using brute force euclidean distance

# Unique weather stations
stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

def find_nearest_station(row):
    dists = np.sqrt((stations['Longitude'] - row['Longitude'])**2 + 
                    (stations['Latitude'] - row['Latitude'])**2) 
    nearest_idx = dists.idxmin() # get index of nearest station (which aligns with the index in stations)
    # return a series with the nearest station info for the current row
    return pd.Series({
        "Facility ID": row['Facility ID'],
        "Facility Name": row['Facility Name'],
        "Longitude": row['Longitude'],
        "Latitude": row['Latitude'],
        'Nearest_Station': stations.loc[nearest_idx, 'Station'],
        'Station_Longitude': stations.loc[nearest_idx, 'Longitude'],
        'Station_Latitude': stations.loc[nearest_idx, 'Latitude']
    })

# apply to each unique facility location
unique_facilities = facility_df[["Facility Name", "Facility ID", "Longitude", "Latitude"]].drop_duplicates()
facility_df = unique_facilities.apply(find_nearest_station, axis=1)
facility_df["Facility ID"] = facility_df["Facility ID"].astype(int)  # ensure Facility ID is int

In [14]:
campd_wy = pd.read_csv("WY/Hourly/Concatted File/wy_2019_2024_complete_cleaned_Full.csv", dtype=campd_type_dict)

In [ ]:
# Merge mapping into campd_wy

campd_wy["datetime"] = pd.to_datetime(campd_wy["datetime"])
campd_wy = campd_wy.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_wy_with_temp = campd_wy.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_wy_with_temp = campd_wy_with_temp.drop(columns=['Station'])


In [17]:
campd_wy_with_temp.to_csv("WY/Hourly/Concatted File/wy_2019_2024_complete_cleaned_Full.csv", index=False)


<h2>ND</h2>

In [38]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/ND')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/ND/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [ ]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})

# remove any duplicate entries (data issues)
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"], keep='first')
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,BIS,2019-01-01 00:52,-100.75,46.77,-17.00
1,ISN,2019-01-01 00:52,-103.64,48.17,-9.00
2,N60,2019-01-01 00:52,-101.44,47.65,-6.00
3,GFK,2019-01-01 00:53,-97.18,47.95,-18.90
4,HEI,2019-01-01 00:53,-102.66,46.01,-14.10
...,...,...,...,...,...
5696707,7L2,2024-12-31 23:55,-100.25,46.22,19.40
5696708,46D,2024-12-31 23:55,-99.15,47.45,14.00
5696709,DVL,2024-12-31 23:56,-98.91,48.11,10.00
5696710,DIK,2024-12-31 23:56,-102.80,46.80,17.00


In [40]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [41]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   5149557.00
mean         41.59
std          24.93
min         -41.00
25%          24.10
50%          42.00
75%          62.00
max         120.20
Name: Temperature (F), dtype: float64

In [42]:
# hottest official temperature recorded in ND is 121, fill with NaN if higher than 120
temp_df.loc[temp_df["Temperature (F)"] > 120, "Temperature (F)"] = np.nan

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [ ]:
# round to the lower hour 
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [45]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [46]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024_ND.csv")

In [ ]:
# calculate the nearest weather station for each facility using brute force euclidean distance

# Unique weather stations
stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

def find_nearest_station(row):
    dists = np.sqrt((stations['Longitude'] - row['Longitude'])**2 + 
                    (stations['Latitude'] - row['Latitude'])**2) 
    nearest_idx = dists.idxmin() # get index of nearest station (which aligns with the index in stations)
    # return a series with the nearest station info for the current row
    return pd.Series({
        "Facility ID": row['Facility ID'],
        "Facility Name": row['Facility Name'],
        "Longitude": row['Longitude'],
        "Latitude": row['Latitude'],
        'Nearest_Station': stations.loc[nearest_idx, 'Station'],
        'Station_Longitude': stations.loc[nearest_idx, 'Longitude'],
        'Station_Latitude': stations.loc[nearest_idx, 'Latitude']
    })

# apply to each unique facility location
unique_facilities = facility_df[["Facility Name", "Facility ID", "Longitude", "Latitude"]].drop_duplicates()
facility_df = unique_facilities.apply(find_nearest_station, axis=1)
facility_df["Facility ID"] = facility_df["Facility ID"].astype(int)  # ensure Facility ID is int

In [ ]:
campd_nd = pd.read_csv("ND/Hourly/Concatted File/nd_2019_2024_complete_cleaned.csv", dtype=campd_type_dict)

In [ ]:
# Merge mapping into campd_nd

campd_nd["datetime"] = pd.to_datetime(campd_nd["datetime"])
campd_nd = campd_nd.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_nd_with_temp = campd_nd.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_nd_with_temp = campd_nd_with_temp.drop(columns=['Station'])

In [64]:
campd_nd_with_temp.to_csv("ND/Hourly/Concatted File/nd_2019_2024_complete_cleaned.csv", index=False)


<h2>MT</h2>

In [3]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/MT')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/MT/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [ ]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})

# remove any duplicate entries (data issues)
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"], keep='first')
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,JDN,2019-01-01 00:48,-106.9475,47.3258,-5.0
1,BHK,2019-01-01 00:51,-104.2595,46.3476,-13.0
2,DLN,2019-01-01 00:53,-112.5525,45.2554,-9.0
3,LVM,2019-01-01 00:53,-110.4500,45.7000,-8.0
4,MLS,2019-01-01 00:53,-105.8863,46.4280,0.0
...,...,...,...,...,...
5820337,1GM,2024-12-31 23:55,-112.1000,47.3300,20.7
5820338,GDV,2024-12-31 23:56,-104.8072,47.1387,18.0
5820339,CTB,2024-12-31 23:56,-112.3761,48.6084,18.0
5820340,BZN,2024-12-31 23:56,-111.1608,45.7881,14.0


In [5]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

count   5780202.00
mean         44.52
std          22.00
min         -45.00
25%          30.00
50%          44.60
75%          60.00
max         109.00
Name: Temperature (F), dtype: float64

In [7]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [ ]:
# round to the lower hour 
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [9]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [10]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024_MT.csv")

In [ ]:
# calculate the nearest weather station for each facility using brute force euclidean distance

# Unique weather stations
stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

def find_nearest_station(row):
    dists = np.sqrt((stations['Longitude'] - row['Longitude'])**2 + 
                    (stations['Latitude'] - row['Latitude'])**2) 
    nearest_idx = dists.idxmin() # get index of nearest station (which aligns with the index in stations)
    # return a series with the nearest station info for the current row
    return pd.Series({
        "Facility ID": row['Facility ID'],
        "Facility Name": row['Facility Name'],
        "Longitude": row['Longitude'],
        "Latitude": row['Latitude'],
        'Nearest_Station': stations.loc[nearest_idx, 'Station'],
        'Station_Longitude': stations.loc[nearest_idx, 'Longitude'],
        'Station_Latitude': stations.loc[nearest_idx, 'Latitude']
    })

# apply to each unique facility location
unique_facilities = facility_df[["Facility Name", "Facility ID", "Longitude", "Latitude"]].drop_duplicates()
facility_df = unique_facilities.apply(find_nearest_station, axis=1)
facility_df["Facility ID"] = facility_df["Facility ID"].astype(int)  # ensure Facility ID is int

In [ ]:
campd_mt = pd.read_csv("MT/Hourly/Concatted File/mt_2019_2024_complete_cleaned.csv", dtype=campd_type_dict)

In [ ]:
# Merge mapping into campd_mt

campd_mt["datetime"] = pd.to_datetime(campd_mt["datetime"])
campd_mt = campd_mt.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_mt_with_temp = campd_mt.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_mt_with_temp = campd_mt_with_temp.drop(columns=['Station'])


In [23]:
campd_mt_with_temp.to_csv("MT/Hourly/Concatted File/mt_2019_2024_complete_cleaned.csv", index=False)
